# Linkedin


In [1]:
pip install crewai==1.14.1 crewai_tools==1.14.1 langchain_community==0.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install numpy

In [4]:
from crewai import Agent, Task, Crew
from crewai.tools import tool

In [ ]:
import openai

openai.api_key = " "

In [5]:
from google.colab import userdata

In [6]:
import os
#from utils import get_openai_api_key

#openai_api_key = get_openai_api_key()
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'

In [8]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

In [9]:
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

## Creating Agents for LinkedIn

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

In [10]:
planner = Agent(
    role="LinkedIn Content Planner",
    goal="Plan engaging and factually accurate content for LinkedIn on {topic}",
    backstory="You're working on planning a LinkedIn post "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
	  verbose=True
)

### Agent: Writer

In [11]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic for LinkedIn: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic for LinkedIn: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements"
              "Make it engaging, use bullet points where needed, and end with a question to encourage comments and also make sure content is not more than 500 words"
              "You are a LinkedIn content expert who can write posts that go viral",
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [12]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a LinkedIn article "
              "from the Content Writer. "
              "Your goal is to review the LinkedIn article "
              "to ensure that it follows  best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible"
              "Fix grammar, improve the flow, make the tone more natural, and ensure it sounds human and engaging"
              "You are a meticulous editor, polishing posts to sound engaging and professional.",

    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [13]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"

    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis "
        " and resources.",
    agent=planner,
)

### Task: Write

In [14]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "LinkedIn post on {topic}.\n"
        "2. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion and content is not more than 500 words\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice and add relevant hashtags.\n"
    ),
    expected_output="A well-structured LinkedIn post with hook, insights, and call-to-action.",
    agent=writer,
)

### Task: Edit

In [15]:
edit = Task(
    description=("Proofread the given LinkedIn post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A final LinkedIn post ready to publish.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [17]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [18]:
result = crew.kickoff(inputs={"topic": "Multi-Agent Framework"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9aca84c3-6372-4950-93ec-166196cb74ee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Multi-Agent Framework.              │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│                                                                                                                 │
│  ID: 39e5d1b7-9eee-446c-80f4-c0fe1ed6511a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: LinkedIn Content Planner                                                                                │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Multi-Agent Framework.              │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'latest trends in Multi-Agent Framework 2023'}                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'key players in Multi-Agent Framework 2023'}                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Use the content plan to craft a compelling LinkedIn post on Multi-Agent Framework.                    │
│  2. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing conclusion  │
│  and content is not more than 500 words                                                                         │
│  5. Proofread for grammatical errors and alignment with the brand's voice and add relevant hashtags.            │
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Proofread the given LinkedIn post for grammatical errors and alignment with the brand's voice.           │
│  ID: 5bc2e9a6-7e85-4779-ae3e-2bcb8fc04cbe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given LinkedIn post for grammatical errors and alignment with the brand's voice.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Harnessing the Power of Multi-Agent Frameworks: The Future of AI Collaboration**                             │
│                                                                                                                 │
│  In a rapidly evolving AI landscape, the embrace of Multi-Agent Frameworks (MAFs) isn't just innovation—it's a  │
│  necessity for maximizing efficiency and promoting collaborative problem-solving. MAFs enable various AI        │
│  agents to collaboratively tackle complex tasks, and their relevance continues to burgeon as organizations      │
│  seek scalable and adaptable solutions.                                                                         │
│                                                                                                                 │
│  ### The Latest Trends in Multi-Agent Frameworks (2023)                                                         │
│                                                                                                                 │
│  1. **Rise of Complex Multi-Agent Systems:**                                                                    │
│     - Emerging frameworks such as **AutoGen** and **LangGraph** represent the next generation of multi-agent    │
│  systems. They integrate generative AI, enhancing task management and facilitating nuanced interactions         │
│  between agents. Companies leveraging these advanced frameworks can unlock improved operational efficiency.     │
│     - For insights into the evolution of these systems, check out articles from                                 │
│  [Dragonscale](https://blog.dragonscale.ai/beyond-single-tasks-the-compelling-case-for-multi-agent-systems)     │
│  and                                                                                                            │
│  [Medium](https://medium.com/@hieutrantrung.it/the-ai-agent-framework-landscape-in-2025-what-changed-and-what-  │
│  matters-3cd9b07ef2c3).                                                                                         │
│                                                                                                                 │
│  2. **Governance and Safety:**                                                                                  │
│     - With the proliferation of multi-agent systems, governance and safety concerns have intensified.           │
│  Implementing robust regulatory frameworks will be crucial to managing the complexities these systems bring.    │
│  Effective governance ensures that the benefits of multi-agent systems outweigh the risks. A comprehensive      │
│  guide can be found at                                                                                          │
│  [Lumenova](https://www.lumenova.ai/blog/taming-complexity-governing-multi-agent-systems-guide/).               │
│                                                                                                                 │
│  ### Key Players Shaping Multi-Agent Frameworks                                                                 │
│                                                                                                                 │
│  - **AutoGen by Microsoft:** Renowned for its focus on conversational agents, enhancing user interaction.       │
│  - **LangChain:** Gaining traction for creating streaml

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given LinkedIn post for grammatical errors and alignment with the brand's voice.           │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the results of your execution as markdown in the notebook.

In [20]:
from IPython.display import Markdown, Image, display

caption = result.raw
image_url = result.tasks_output[0].raw

display(Image(url=image_url))
display(Markdown(caption))

**Harnessing the Power of Multi-Agent Frameworks: The Future of AI Collaboration** 

In a rapidly evolving AI landscape, the embrace of Multi-Agent Frameworks (MAFs) isn't just innovation—it's a necessity for maximizing efficiency and promoting collaborative problem-solving. MAFs enable various AI agents to collaboratively tackle complex tasks, and their relevance continues to burgeon as organizations seek scalable and adaptable solutions. 

### The Latest Trends in Multi-Agent Frameworks (2023)

1. **Rise of Complex Multi-Agent Systems:**
   - Emerging frameworks such as **AutoGen** and **LangGraph** represent the next generation of multi-agent systems. They integrate generative AI, enhancing task management and facilitating nuanced interactions between agents. Companies leveraging these advanced frameworks can unlock improved operational efficiency.
   - For insights into the evolution of these systems, check out articles from [Dragonscale](https://blog.dragonscale.ai/beyond-single-tasks-the-compelling-case-for-multi-agent-systems) and [Medium](https://medium.com/@hieutrantrung.it/the-ai-agent-framework-landscape-in-2025-what-changed-and-what-matters-3cd9b07ef2c3).

2. **Governance and Safety:**
   - With the proliferation of multi-agent systems, governance and safety concerns have intensified. Implementing robust regulatory frameworks will be crucial to managing the complexities these systems bring. Effective governance ensures that the benefits of multi-agent systems outweigh the risks. A comprehensive guide can be found at [Lumenova](https://www.lumenova.ai/blog/taming-complexity-governing-multi-agent-systems-guide/).

### Key Players Shaping Multi-Agent Frameworks

- **AutoGen by Microsoft:** Renowned for its focus on conversational agents, enhancing user interaction.
- **LangChain:** Gaining traction for creating streamlined agent workflows.
- **Consortiums like Deloitte and Accenture:** Leading consultants on implementation strategies for businesses looking to leverage multi-agent systems.

For a detailed comparison of frameworks, don’t miss the [Medium article](https://viveksinghpathania.medium.com/top-5-multi-agent-ai-frameworks-powering-the-future-of-intelligent-applications-7affbb458cb1).

### Noteworthy Innovations

- Microsoft’s Agent Framework has recently unveiled enhancements that significantly impact sectors like finance, particularly in multi-agent fraud detection. The implications of such advancements can be explored further in this [Medium piece](https://medium.com/@iamsalmankarim/microsoft-agent-framework-the-unified-ai-engine-fintech-has-been-waiting-for-350eea2906a8).
- Innovations in agent cooperation strategies highlight the potential of AI agents to perform complex tasks effectively. Get the latest insights on these advancements in a discussion from [InfoWorld](https://www.infoworld.com/article/4154335/multi-agent-ai-is-the-new-microservices.html).

### Real-World Applications

As businesses integrate multi-agent frameworks, numerous case studies showcase their significant impacts. From improved decision-making to enhanced efficiency metrics, the successful applications of these frameworks demonstrate their potential to revolutionize traditional business practices.

### Call to Action

Now is the time for professionals and decision-makers to delve into the world of multi-agent frameworks. By adapting MAFs tailored to your specific business needs, you can harness their full potential. Ready to take the plunge? Explore further learnings and tools available in our resources, and start shaping your multi-agent strategy today!

How do you see Multi-Agent Frameworks transforming your industry? Share your insights below! 

#AI #MultiAgentFramework #Collaboration #ArtificialIntelligence #TechnologyInnovation #BusinessStrategy #GenerativeAI #Automation